"""
NOTEBOOK BOUNDARY RULES:
- This notebook is observational only
- No hypotheses may be defined here
- No statistical conclusions may be made
- No SQL allowed
- No modeling allowed
"""

# 03 Stability Analysis

In [ ]:
# INTENT: STRUCTURAL_CONTEXT
# LAYER: structural
# SOURCE_CONTRACT: analysis/ANALYSIS_INTENT_SPEC.md
# ALLOWED_INPUTS: analysis/dal/*
# FORBIDDEN: SQL, feature engineering, pipeline logic, model logic

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT / "analysis").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from analysis.curated.player_gameweek_spine import build_player_gameweek_spine
from analysis.state.player_gameweek_state import build_player_gameweek_state
from fpl_intelligence.config import DB_PATH

df_spine = build_player_gameweek_spine(DB_PATH)
df_state = build_player_gameweek_state(df_spine)

df = df_state.merge(
    df_spine[["player_id", "gameweek", "total_points"]],
    on=["player_id", "gameweek"],
    how="inner"
)

total_row_count = len(df)


## 1. Mean vs Variance Comparison

In [ ]:
(
    df.groupby("fixture_context", dropna=False)["total_points"]
    .agg(["mean", "std"])
    .assign(coefficient_of_variation=lambda x: x["std"] / x["mean"])
    .reset_index()
)

In [ ]:
(
    df.groupby("fixture_count", dropna=False)["total_points"]
    .agg(["mean", "std"])
    .assign(coefficient_of_variation=lambda x: x["std"] / x["mean"])
    .reset_index()
)

In [ ]:
(
    df.groupby("home_away_profile", dropna=False)["total_points"]
    .agg(["mean", "std"])
    .assign(coefficient_of_variation=lambda x: x["std"] / x["mean"])
    .reset_index()
)

## 2. Sample Size Sensitivity

In [ ]:
(
    df.groupby("fixture_context", dropna=False)
    .size()
    .rename("count")
    .to_frame()
    .assign(percentage_of_total_rows=lambda x: x["count"] / total_row_count * 100)
    .reset_index()
)

In [ ]:
(
    df.groupby("fixture_count", dropna=False)
    .size()
    .rename("count")
    .to_frame()
    .assign(percentage_of_total_rows=lambda x: x["count"] / total_row_count * 100)
    .reset_index()
)

In [ ]:
(
    df.groupby("home_away_profile", dropna=False)
    .size()
    .rename("count")
    .to_frame()
    .assign(percentage_of_total_rows=lambda x: x["count"] / total_row_count * 100)
    .reset_index()
)

## 3. Distribution Spread

In [ ]:
(
    df.groupby("fixture_context", dropna=False)["total_points"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "p25", 0.5: "p50", 0.75: "p75"})
    .reset_index()
)

In [ ]:
(
    df.groupby("fixture_count", dropna=False)["total_points"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "p25", 0.5: "p50", 0.75: "p75"})
    .reset_index()
)

In [ ]:
(
    df.groupby("home_away_profile", dropna=False)["total_points"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "p25", 0.5: "p50", 0.75: "p75"})
    .reset_index()
)

## 4. Gameweek Stability Check

In [ ]:
(
    df.groupby(["gameweek", "fixture_context"], dropna=False)["total_points"]
    .mean()
    .reset_index(name="mean_total_points")
)

In [ ]:
(
    df.groupby(["gameweek", "fixture_context"], dropna=False)["total_points"]
    .mean()
    .reset_index(name="mean_total_points")
    .groupby("fixture_context", dropna=False)["mean_total_points"]
    .agg(["var", "count", "mean", "std"])
    .reset_index()
)

## 5. Extreme Value Influence

In [ ]:
(
    df.groupby("fixture_context", dropna=False)["total_points"]
    .agg(original_mean="mean", top_1pct_threshold=lambda s: s.quantile(0.99))
    .reset_index()
    .merge(
        df.groupby("fixture_context", dropna=False)
        .apply(lambda g: g.loc[g["total_points"] <= g["total_points"].quantile(0.99), "total_points"].mean(), include_groups=False)
        .rename("mean_excluding_top_1pct")
        .reset_index(),
        on="fixture_context",
        how="left"
    )
)

In [ ]:
(
    df.groupby("fixture_count", dropna=False)["total_points"]
    .agg(original_mean="mean", top_1pct_threshold=lambda s: s.quantile(0.99))
    .reset_index()
    .merge(
        df.groupby("fixture_count", dropna=False)
        .apply(lambda g: g.loc[g["total_points"] <= g["total_points"].quantile(0.99), "total_points"].mean(), include_groups=False)
        .rename("mean_excluding_top_1pct")
        .reset_index(),
        on="fixture_count",
        how="left"
    )
)

In [ ]:
(
    df.groupby("home_away_profile", dropna=False)["total_points"]
    .agg(original_mean="mean", top_1pct_threshold=lambda s: s.quantile(0.99))
    .reset_index()
    .merge(
        df.groupby("home_away_profile", dropna=False)
        .apply(lambda g: g.loc[g["total_points"] <= g["total_points"].quantile(0.99), "total_points"].mean(), include_groups=False)
        .rename("mean_excluding_top_1pct")
        .reset_index(),
        on="home_away_profile",
        how="left"
    )
)